### Imports

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


### Load and prepare the data

In [2]:
transform = transforms.ToTensor() # Python Imaging Library (PIL) --> float tensor, scales pixels from [0, 255] --> [0.0, 1.0], reshapes from (28, 28) --> (1, 28, 28)

In [4]:
train_dataset = datasets.MNIST('./data', train=True,  download=True, transform=transforms.ToTensor())
test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.86MB/s]

Training images: 60000
Test images: 10000


In [5]:
train = DataLoader(train_dataset, batch_size=64, shuffle=True) # shuffle randomises the order of images each epoch
test = DataLoader(test_dataset,  batch_size=64)

### Building the model

You could do the long way:

In [6]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden  = nn.Linear(784, 30)
        self.output  = nn.Linear(30, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.hidden(x))
        x = self.output(x)
        return x

or use `nn.Sequential` and be much faster:

In [7]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 30),
    nn.ReLU(),
    nn.Linear(30, 10),
)

In [8]:
print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=30, bias=True)
  (2): ReLU()
  (3): Linear(in_features=30, out_features=10, bias=True)
)


### Training

In [9]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

for epoch in range(5):
    for images, labels in train_loader:
        prediction = model.forward(images)
        loss = criterion(prediction, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/5 — loss: {loss.item():.4f}')

Epoch 1/5 — loss: 0.1740
Epoch 2/5 — loss: 0.1372
Epoch 3/5 — loss: 0.1319
Epoch 4/5 — loss: 0.0849
Epoch 5/5 — loss: 0.0429


In [10]:
test_loader = DataLoader(test_dataset, batch_size=64)

correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        correct += (model(images).argmax(1) == labels).sum().item()

print(f"Test accuracy: {correct / len(test_dataset):.2%}")

Test accuracy: 95.71%
